In [2]:
from elasticsearch import Elasticsearch

client = Elasticsearch(
    hosts=["http://localhost:9200"]
)

print(client.info())

{'name': 'dbfbabaeac35', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'EUXoudxUR6W8O2Gd8ifMXA', 'version': {'number': '8.12.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '6185ba65d27469afabc9bc951cded6c17c21e3f3', 'build_date': '2024-02-01T13:07:13.727175297Z', 'build_snapshot': False, 'lucene_version': '9.9.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [8]:
query = {'terms': {'_id': ['395d26db-4d1f-4574-8412-46f96e242401']}}

response = client.search(
    index="api-test", query=query,
    _source={"excludes": ["vector"]}
)
response

ObjectApiResponse({'took': 5, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'api-test', '_id': '395d26db-4d1f-4574-8412-46f96e242401', '_score': 1.0, '_source': {'text': 'Q: Wann muss man die Zieletage in seillosen Aufzügen auswählen? \nA: vor Fahrtantritt (d.\xa0h. noch außerhalb des Aufzug)\nTopic: Aufzugsanlage', 'metadata': {'process_id': 'Slice_0_100', 'gdpr_id': '40368', 'topic': 'Wann muss man die Zieletage in seillosen Aufzügen auswählen? ', 'type': 'answer', 'len': 1892, 'chunk_id': 0, 'chunks': 4}}}]}})

In [ ]:
response['hits']['hits'][0]['_source']


In [3]:
from mylibs.embedding.embedding import get_vectorstore

question = "What is the capital of France?"
filter = {"bool": {"must": [{"term": {"metadata.type": "answer"}}]}}
vectorstore = get_vectorstore()
retriever = vectorstore.as_retriever(search_kwargs=filter)

documents = retriever.invoke(
    question,
)


In [4]:
documents

[Document(metadata={'process_id': 'Slice_0_100', 'gdpr_id': '36799', 'topic': 'Wo leben die meisten schwarzen in Frankreich?', 'type': 'answer', 'len': 3314, 'chunk_id': 0, 'chunks': 5}, page_content='Q: Wo leben die meisten schwarzen in Frankreich?\nA: Hauptstadtregion Île-de-France und im Großraum Marseille\nTopic: Schwarze'),
 Document(metadata={'process_id': 'Slice_0_100', 'gdpr_id': '36798', 'topic': 'Warum gab es in der französischen Regierung oft schwarze Minister?', 'type': 'answer', 'len': 3340, 'chunk_id': 2, 'chunks': 5}, page_content="Die massenhafte Einwanderung von Schwarzafrikanern in das französische Mutterland begann nach der Dekolonisierung in den 1960er Jahren. Die Mehrheit der schwarzen Franzosen lebt in der Hauptstadtregion Île-de-France und im Großraum Marseille. Kritiker verweisen auf die ethnische Segregation der Bevölkerung: In den ''banlieues'' der Großstädte, rund um Paris vor allem in den Départements Val d'Oise und Seine-Saint-Denis, leben afrikanische und 